In [1]:
import sys
sys.path.append('/workspaces/BlizzardX')

In [2]:
from src.config.config_manager import ConfigManager
from src.ghcn_daily.ghcn_data_handler import GHCNDataHandler
from src.ghcn_daily.data_fetch import DataFetcher
from src.ghcn_daily.data_processing_old import WeatherDataProcessor
import numpy as np

In [3]:
ghcn = GHCNDataHandler()
config = ConfigManager(config_directory='/workspaces/BlizzardX/src/config')
config.load_config('settings.json')
data_fetcher = DataFetcher(config_file="settings.json",data_type='dataframe')

In [4]:
stations= ghcn.get_station_data(config.get('settings.json', 'data_sources.stations'))
inventory = ghcn.get_inventory_data(config.get('settings.json', 'data_sources.inventory'))

In [5]:
s_state_list=stations[stations['STATE']=='NH']['ID'].tolist()
s_live_list=inventory[(inventory['ID'].isin(s_state_list)) & (inventory['LASTYEAR']>2024) & (inventory['FIRSTYEAR']<2015)]['ID'].unique().tolist()

In [6]:
data= await data_fetcher.save_data(s_live_list)

Fetching Data: 100%|██████████████████████████████████████████████████| 9/9 [00:15<00:00,  1.74s/it]


CPU usage is high! Decreasing workers to 4
CPU usage is high! Decreasing workers to 2
CPU usage is stable. Increasing workers to 4
CPU usage is stable. Increasing workers to 6


In [7]:
flag_columns = [col for col in data.columns if 'FLAG' in col]
data= data.drop(columns=flag_columns)
#data.replace(-9999.0, np.nan, inplace=True)
weather_variables = ['TMAX', 'TMIN', 'SNOW', 'SNWD', 'PRCP']

In [8]:
from src.ghcn_daily.data_processing import WeatherDataTransformer,WeatherDataImputer
processor = WeatherDataTransformer(data, weather_variables)

In [9]:
df=processor.process_data()

In [10]:
df.replace(-9999.0, np.nan, inplace=True)
df.replace(-999.9, np.nan, inplace=True)

In [11]:
import pandas as pd
df = pd.merge(df, stations, on='ID', how='left')
df = df[['DATE','ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'Season', 'TMIN', 'TMAX', 'PRCP', 'SNOW', 'SNWD']]

In [12]:
from src.ghcn_daily.data_filtering import WeatherDataFilter
filter = WeatherDataFilter(df)

In [13]:
df=df[df['ID'].isin(filter.get_stations_with_zero_missing_dates())]


In [14]:
df=df[df['ID'].isin(filter.get_stations_with_low_missing_values(threshold=5))]

In [15]:
import os
os.makedirs(os.path.dirname('/workspaces/BlizzardX/Data/processed_data.csv'), exist_ok=True)

In [16]:
df.to_csv('/workspaces/BlizzardX/Data/processed_data.csv', index=False)

In [17]:
imputer= WeatherDataImputer(df)

In [18]:
df=imputer.clean_all()

In [19]:
df.to_csv('/workspaces/BlizzardX/Data/cleaned_data.csv', index=False)